# SimEX Loop
We will be asked to create a "coach" loop that organize the training of a controller, which will do the following 

The following is pseudo-code which shall not necessarily represent the real method signatures, but shall indicate the flow of logic, presenting where which information is used and consumed.  
```bash
init trainingData 

counter = 0 

newTrainingData = {} 

Do 

counter++ 

trainingData = trainingData + newTrainingData 

controller.doTraining(trainingData) 

PathtoCSV= runSimEx(controller) //creates new csv outputs 
 - Seems to only be the simulator? providing new function in the simulator as parameter (retrained model?) 

newTrainingData = checkPerformance(PathToReference, PathToCSV).  

// method provide by Martin, reading csv output from simex, but is a simple loop testing for fixed areas if their performance is higher or lower than the reference (computing the value of the provide polynom, identifiyable via the intervals in the csv) and if the currentPerfomance is worse than reference add datapoints for new training data (approximate at least reference values as targets)).   

while(newTrainingData !={} AND counter<MaxIteration)
```
methods in need to be provided by HR side, but they need our help to give them a structure to work in 

Note, with this loop you can theoretically implement zero learning (from untrained to hero automatically 

Note 2: this loops enable the interesting question, when do we stop learning, even though results are not perfect, e.g. when we have to acknowledge given available actions, we cannot solve the problem (as for the moment, what is "learned" is the action-selection, while primitive actions are pre-defined. I.e. when the set of actions is not sufficient no learning can be successful, an alternative when to stopp before I overfit... * all of this are interesting research questions (for later work) 

In [1]:
import os
import sys

# Add repo root (for simex package) and this example dir (for marl package)
# Run this notebook from examples/marl_vsl/ so os.getcwd() resolves correctly
_example_dir = os.getcwd()
_repo_root = os.path.abspath(os.path.join(_example_dir, "../.."))
sys.path.insert(0, _repo_root)
sys.path.insert(0, _example_dir)

In [2]:
import pandas as pd
import numpy as np

from marl.performance import automatic_performance
from simex.components.validator import Validator
from simex.components.modifier import Modifier
from simex import Simex
from marl.training import Controller
from marl_simulator import Simulator

base_file = os.path.join(_example_dir, "base_file", "simex_output-MARL_novsl-20250102-201414.csv")
vsl_file = os.path.join(_example_dir, "vsl_file", "simex_output-MARL_vsl_kresimir_data-20250107-090617.csv")

In [3]:
 # Standalone analysis cell: compare a pre-existing VSL output against the baseline.
# Set vsl_file to the path of a SimEx VSL output CSV before running this cell.
if vsl_file is None:
    print("vsl_file is not set — skipping. Set vsl_file to a SimEx VSL output CSV path.")
else:
    df_baseline = pd.read_csv(base_file)
    df_control = pd.read_csv(vsl_file)
    dataset_control = df_control.to_numpy()
    dataset_baseline = df_baseline.to_numpy()

    bad_regions = automatic_performance(
        dataset_baseline, dataset_control,
        incremnet_step_for_x=10,
        max_order_of_polynom=9,
        tolerance_in_diffrence=12
    )
    print("Bad regions:", bad_regions)
    if bad_regions:
        newTrainingData = [item for i, j in bad_regions for item in np.arange(i, j, 5)]
        print("New training data:", newTrainingData)

INTERVALS: [[3160.0, 3170.0], [3250.0, 3260.0], [3260.0, 3270.0], [3270.0, 3280.0], [3280.0, 3290.0], [3290.0, 3300.0], [3300.0, 3310.0], [3310.0, 3320.0], [3320.0, 3330.0], [3360.0, 3370.0], [3370.0, 3380.0], [3380.0, 3390.0], [3390.0, 3400.0], [3430.0, 3440.0], [3460.0, 3470.0], [3470.0, 3480.0], [3480.0, 3490.0], [3490.0, 3500.0], [3500.0, 3510.0], [3510.0, 3520.0], [3520.0, 3530.0], [3530.0, 3540.0], [3540.0, 3550.0], [3550.0, 3560.0], [3560.0, 3570.0], [3570.0, 3580.0], [3580.0, 3590.0], [3590.0, 3600.0], [3600.0, 3610.0], [3610.0, 3620.0], [3620.0, 3630.0], [3630.0, 3640.0], [3640.0, 3650.0], [3650.0, 3660.0], [3660.0, 3670.0], [3700.0, 3710.0], [3730.0, 3740.0], [3740.0, 3750.0], [3780.0, 3790.0], [3790.0, 3800.0], [3800.0, 3810.0], [3810.0, 3820.0], [3820.0, 3830.0], [3830.0, 3840.0], [3840.0, 3850.0], [3850.0, 3860.0], [3860.0, 3870.0], [3890.0, 3900.0], [3900.0, 3910.0], [3910.0, 3920.0], [3920.0, 3930.0], [3930.0, 3940.0], [3940.0, 3950.0], [3950.0, 3960.0], [3960.0, 3970.0]

In [ ]:
import os
import pandas as pd
import numpy as np

from marl.performance import automatic_performance
from simex.components.validator import Validator
from simex.components.modifier import Modifier
from simex import Simex
from marl.training import Controller
from marl_simulator import Simulator

base_file = os.path.join(_example_dir, "base_file", "simex_output-MARL_novsl-20250102-201414.csv")
results_path = os.getenv('MARL_RESULTS_PATH', os.path.join(_example_dir, '../../marl_training_file/'))
iterations_of_training = 100
os.environ['MARL_PATH_TRAIN'] = results_path

df_baseline = pd.read_csv(base_file)
dataset_baseline = df_baseline.to_numpy()
counter = 0
MaxIteration = 5
newTrainingData = []
trainingData = [i for i in range(2500, 4100, 100)]
load_trained_controller = True
start_training = 6000

while True:
    counter += 1
    if counter > 1 and not load_trained_controller:
        trainingData = trainingData + newTrainingData
        controller = Controller(results_path)
        controller.marl_vsl_training(trainingData)
    elif counter > 1 and load_trained_controller:
        print(f"============= New training data {newTrainingData}")
        trainingData = newTrainingData
        controller = Controller(results_path)
        last_run = controller.marl_vsl_training(trainingData, run_start=start_training, iterations=iterations_of_training, vsl_on=1)
        print(f"Training ended at run {last_run}")
        os.environ['END_RUN_MARL'] = str(start_training + iterations_of_training)
        start_training = start_training + iterations_of_training

    simex_loop_vsl = Simex(instance_name=f"VSL_marl_loop-{counter}", smoothen=True)
    control_file = simex_loop_vsl.run_simex(
        simulator_function=Simulator.marl_vsl_simulator,
        modifier=Modifier.modifierA,
        validator=Validator.local_exploration_validator_A,
        parallel=False
    )
    df_control = pd.read_csv(control_file)
    dataset_control = df_control.to_numpy()

    bad_regions = automatic_performance(
        dataset_baseline, dataset_control,
        incremnet_step_for_x=10,
        max_order_of_polynom=9,
        tolerance_in_diffrence=12
    )
    print("Bad regions:", bad_regions)
    if bad_regions:
        newTrainingData = [item for i, j in bad_regions for item in np.arange(i, j, 5)]
        print("New training data:", newTrainingData)

    if not newTrainingData or counter > MaxIteration:
        break

os.environ['END_RUN_MARL'] = str(6000)

Instance name VSL_marl_loop-1
Results dir /Users/amy/tmp/repos/SimEx/examples/marl_vsl/results_dir_VSL_marl_loop-1-20260410-094838
Temp x: [2600, 2700, 2800, 2900, 3000, 3100, 3200, 3300, 3400, 3500, 3600, 3700, 3800, 3900]
MAIN mod outcome ([[2600.0, 2681.5384615384614, 2766.153846153846, 2853.846153846154, 2944.6153846153848, 3038.4615384615386, 3135.3846153846152, 3235.3846153846152, 3338.4615384615386, 3444.6153846153848, 3553.846153846154, 3666.153846153846, 3781.5384615384614, 3900.0]], [[2500, 4000]])
AG SIM 0
Results path: /Users/amy/tmp/repos/SimEx/examples/marl_vsl/../../marl_training_file/ and model /Users/amy/tmp/repos/SimEx/marl_model_MD//
SIM RUN training phase 6000
 Retrying in 1 seconds


Step #5400.25 (1ms ~= 250.00*RT, ~215000.00UPS, TraCI: 20ms, vehicles TOT 5639 ACT 215 BUF #4500.00 (1ms ~= 250.00*RT, ~235000.00UPS, TraCI: 0ms, vehicles TOT 4850 ACT 235 BUF 
AG SIM 0
Results path: /Users/amy/tmp/repos/SimEx/examples/marl_vsl/../../marl_training_file/ and model /Users/amy/tmp/repos/SimEx/marl_model_MD//
SIM RUN training phase 6000
 Retrying in 1 seconds
